# 9.5 Specifying Data Using NumPy Arrays

NumPy is the foundational library for numerical computing in Python. Its central data structure, the `ndarray`, represents dense arrays of uniform type and supports fast, vectorised operations over them. In optimization modelling, NumPy arrays arise naturally whenever the data for a two-dimensional AMPL parameter can be arranged as a full matrix—that is, when every `(row, column)` combination has a defined value.

The approach taken in this section is to build the matrix as a NumPy array, then wrap it in a Pandas DataFrame to attach the row and column labels that correspond to AMPL's index sets. The resulting labeled DataFrame can be assigned to an AMPL parameter directly, just as in [Section 9.4](9_4_data_in_pandas.ipynb). The benefit of using NumPy for the raw matrix is that it makes the numerical structure explicit and allows any NumPy operation—scaling, masking, linear-algebraic transformation—to be applied before the data is passed to AMPL.

In [1]:
# install dependencies
%pip install -q amplpy pandas numpy polars

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Note: you may need to restart the kernel to use updated packages.
Licensed to AMPL Community Edition License for <uykun96@gmail.com>.


We use the same diet model as in previous sections. In addition to loading the model, we set up all one-dimensional data so that the only remaining task is to assign the nutrient-content matrix `amt`.

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;
set LINKS within (NUTR cross FOOD);

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];
param amt {NUTR,FOOD} >= 0;

Overwriting diet.mod


In [3]:
ampl.read("diet.mod")

In [ ]:
import pandas as pd

# Define food items and nutrients
foods = ["BEEF", "CHK", "FISH", "HAM", "MCH", "MTL", "SPG", "TUR"]
nutrs = ["A", "C", "B1", "B2", "NA", "CAL"]

# One-dimensional data as DataFrames
food_df = pd.DataFrame(
    [(3.59, 2, 10), (2.59, 2, 10), (2.29, 2, 10), (2.89, 2, 10),
     (1.89, 2, 10), (1.99, 2, 10), (1.99, 2, 10), (2.49, 2, 10)],
    index=foods, columns=["cost", "f_min", "f_max"]
)
nutr_df = pd.DataFrame(
    [(700, 20000), (700, 20000), (700, 20000), (700, 20000), (0, 50000), (16000, 24000)],
    index=nutrs, columns=["n_min", "n_max"]
)

ampl.set_data(food_df, "FOOD")
ampl.set_data(nutr_df, "NUTR")

## 9.5.1 Representing a Parameter Matrix with NumPy

The parameter `amt {NUTR, FOOD}` assigns a numerical value to every `(nutrient, food)` combination, forming a 6 × 8 matrix. Writing this as a nested Python list is the simplest starting point, but NumPy's `np.array()` makes the matrix structure more explicit and immediately enables element-wise operations if any preprocessing is required.

The rows of the matrix correspond to nutrients and the columns to food items. After constructing the array, it is wrapped in a Pandas DataFrame using `food_df.index` and `nutr_df.index` as the column and row labels respectively, establishing the connection to the AMPL index sets:

In [ ]:
import numpy as np
import pandas as pd

# Nutrient-content matrix: rows = nutrients (NUTR), columns = food items (FOOD)
amt_matrix = np.array([
    [60,  8,   8,  40,  15,  70,  25,  60],   # A
    [20,  0,  10,  40,  35,  30,  50,  20],   # C
    [10, 20,  15,  35,  15,  15,  25,  15],   # B1
    [15, 20,  10,  10,  15,  15,  15,  10],   # B2
    [928, 2180, 945, 278, 1182, 896, 1329, 1397],  # NA
    [295, 770, 440, 430, 315, 400, 379, 450],  # CAL
])

# Wrap in a labeled DataFrame to connect rows/columns to AMPL index sets
amt_df = pd.DataFrame(
    amt_matrix,
    columns=food_df.index.to_list(),   # FOOD set as columns
    index=nutr_df.index.to_list()      # NUTR set as rows
)

# Define LINKS and assign the amt parameter
ampl.set["LINKS"] = [(nutr, food) for nutr in nutr_df.index for food in food_df.index]
ampl.param["amt"] = amt_df

In this representation, each row of `amt_matrix` corresponds to one nutrient and each column to one food item, mirroring the natural matrix layout of the data. The row and column labels, taken from `nutr_df.index` and `food_df.index`, ensure that AMPL receives the values with the correct index associations.

Assigning a full DataFrame to `ampl.param["amt"]` is equivalent to iterating over all `(nutrient, food)` pairs and setting each value individually, but it is far more concise and faster for large matrices. Note that in this example `LINKS` is defined to include all nutrient–food pairs, since every entry of the matrix is well-defined; in problems where some combinations are meaningless or absent, a filter condition can be used when constructing `LINKS` to exclude zero or missing entries, as shown in Section 9.4.